# 17 反向传播深入 - 链式法则与数值示例

反向传播（Backpropagation）是神经网络训练的**核心机制**。本教程将通过详细的数值示例，带你一步步理解反向传播是如何工作的。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("导入库成功！")

## 一、链式法则（Chain Rule）——反向传播的数学基础

### 什么是链式法则？

假设有一个复合函数 $y = f(g(x))$，那么 $y$ 对 $x$ 的导数是：

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

对于多层复合函数 $L = f(z) = f(g(h(x)))$：

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial f} \cdot \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial h} \cdot \frac{\partial h}{\partial x}$$

### 在神经网络中的应用

神经网络的每一层都是前一层的函数：

$$L \xleftarrow{} \hat{y} \xleftarrow{} z^{[L]} \xleftarrow{} a^{[L-1]} \xleftarrow{} z^{[L-1]} \xleftarrow{} \cdots \xleftarrow{} x$$

反向传播就是**从输出到输入**，逐层应用链式法则计算梯度的过程。

In [ ]:
# 简单的链式法则数值演示
print("=== 链式法则数值演示 ===")
print()

# 假设: L = (y - target)^2, y = w * x + b
# 求: dL/dw, dL/db

x = 3.0
w = 2.0
b = 1.0
target = 10.0

# 前向传播
y = w * x + b  # y = 2*3 + 1 = 7
L = (y - target)**2  # L = (7-10)^2 = 9

print(f"前向传播:")
print(f"  x = {x}, w = {w}, b = {b}")
print(f"  y = w*x + b = {w}*{x} + {b} = {y}")
print(f"  L = (y - target)^2 = ({y} - {target})^2 = {L}")
print()

# 反向传播（链式法则）
# dL/dy = 2*(y - target)
# dy/dw = x
# dy/db = 1
# dL/dw = dL/dy * dy/dw = 2*(y-target) * x
# dL/db = dL/dy * dy/db = 2*(y-target) * 1

dL_dy = 2 * (y - target)
dy_dw = x
dy_db = 1.0

dL_dw = dL_dy * dy_dw
dL_db = dL_dy * dy_db

print(f"反向传播（链式法则）:")
print(f"  dL/dy = 2*(y - target) = 2*({y} - {target}) = {dL_dy}")
print(f"  dy/dw = x = {x}")
print(f"  dy/db = 1")
print()
print(f"  dL/dw = dL/dy * dy/dw = {dL_dy} * {x} = {dL_dw}")
print(f"  dL/db = dL/dy * dy/db = {dL_dy} * 1 = {dL_db}")
print()

# 用PyTorch自动求导验证
x_t = torch.tensor(3.0)
w_t = torch.tensor(2.0, requires_grad=True)
b_t = torch.tensor(1.0, requires_grad=True)
target_t = torch.tensor(10.0)

y_t = w_t * x_t + b_t
L_t = (y_t - target_t)**2
L_t.backward()

print(f"PyTorch自动求导验证:")
print(f"  dL/dw = {w_t.grad.item()} (手动计算: {dL_dw})")
print(f"  dL/db = {b_t.grad.item()} (手动计算: {dL_db})")

## 二、完整神经网络的反向传播（含详细数值）

让我们考虑一个包含隐藏层的完整神经网络，并手动计算反向传播的每一步。

### 网络结构

```
输入层 (2个节点) → 隐藏层 (2个节点, sigmoid) → 输出层 (1个节点, 线性)
```

### 前向传播公式

隐藏层：
$$z^{[1]} = W^{[1]}x + b^{[1]}$$
$$a^{[1]} = \sigma(z^{[1]}) = \frac{1}{1 + e^{-z^{[1]}}}$$

输出层：
$$z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}$$
$$\hat{y} = z^{[2]}$$

损失：
$$L = \frac{1}{2}(\hat{y} - y)^2$$

In [ ]:
# 完整的手动反向传播数值示例
print("=" * 70)
print("完整神经网络反向传播 - 详细数值示例")
print("=" * 70)
print()

# === 网络参数（给定值） ===
W1 = np.array([[0.5, -0.3],
               [-0.2, 0.8]])  # shape: (2, 2)
b1 = np.array([0.1, -0.1])     # shape: (2,)
W2 = np.array([0.4, 0.6])      # shape: (2,)
b2 = 0.2                        # scalar

# === 训练数据 ===
x = np.array([1.0, 0.5])       # 输入
y_true = 0.8                    # 真实标签

print("网络参数:")
print(f"  W1 = \n{W1}")
print(f"  b1 = {b1}")
print(f"  W2 = {W2}")
print(f"  b2 = {b2}")
print()
print(f"输入: x = {x}")
print(f"目标: y = {y_true}")
print()

# === 前向传播 ===
print("=" * 50)
print("前向传播")
print("=" * 50)

# 隐藏层
z1 = W1 @ x + b1
a1 = 1 / (1 + np.exp(-z1))  # sigmoid

print(f"隐藏层:")
print(f"  z1 = W1·x + b1 = {z1}")
print(f"  a1 = σ(z1) = {a1}")
print()

# 输出层
z2 = W2 @ a1 + b2
y_pred = z2  # 线性输出

print(f"输出层:")
print(f"  z2 = W2·a1 + b2 = {z2:.6f}")
print(f"  预测值 ŷ = {y_pred:.6f}")
print()

# 损失
loss = 0.5 * (y_pred - y_true)**2
print(f"损失:")
print(f"  L = 0.5*(ŷ - y)² = 0.5*({y_pred:.6f} - {y_true})² = {loss:.6f}")
print()

# === 反向传播 ===
print("=" * 50)
print("反向传播")
print("=" * 50)

# 输出层梯度
dz2 = y_pred - y_true  # dL/dz2 = ŷ - y
print(f"输出层梯度:")
print(f"  dz2 = dL/dz2 = ŷ - y = {y_pred:.6f} - {y_true} = {dz2:.6f}")
print()

dW2 = dz2 * a1  # dL/dW2 = dz2 * a1
db2 = dz2       # dL/db2 = dz2
print(f"  dW2 = dz2 * a1 = {dz2:.6f} * {a1} = {dW2}")
print(f"  db2 = dz2 = {db2:.6f}")
print()

# 隐藏层梯度
da1 = dz2 * W2  # dL/da1 = dz2 * W2
dz1 = da1 * a1 * (1 - a1)  # dL/dz1 = da1 * σ'(z1)

print(f"隐藏层梯度:")
print(f"  da1 = dz2 * W2 = {dz2:.6f} * {W2} = {da1}")
print(f"  σ'(z1) = a1*(1-a1) = {a1*(1-a1)}")
print(f"  dz1 = da1 * σ'(z1) = {da1} * {a1*(1-a1)} = {dz1}")
print()

dW1 = np.outer(dz1, x)  # dL/dW1 = dz1 ⊗ x
db1 = dz1               # dL/db1 = dz1
print(f"  dW1 = dz1 ⊗ x = \n{dW1}")
print(f"  db1 = dz1 = {db1}")
print()

# === 参数更新 ===
lr = 0.5
print(f"参数更新 (学习率 η = {lr}):")
print(f"  W1_new = W1 - η*dW1 = \n{W1} - {lr}*\n{dW1} = \n{W1 - lr*dW1}")
print(f"  b1_new = b1 - η*db1 = {b1} - {lr}*{db1} = {b1 - lr*db1}")
print(f"  W2_new = W2 - η*dW2 = {W2} - {lr}*{dW2} = {W2 - lr*dW2}")
print(f"  b2_new = b2 - η*db2 = {b2} - {lr}*{db2:.6f} = {b2 - lr*db2:.6f}")

## 三、PyTorch的自动求导机制

PyTorch使用**计算图（Computational Graph）**和**自动微分（Autograd）**来自动计算梯度。

### 计算图是什么？

计算图是一个有向无环图（DAG），其中：
- **节点**：输入张量和操作
- **边**：数据流向和梯度传播路径

### Autograd如何工作？

1. **前向传播**时记录所有操作，构建计算图
2. **反向传播**时自动从输出往输入计算梯度
3. 梯度存储在 `.grad` 属性中

In [ ]:
# PyTorch自动求导机制演示
print("=== PyTorch Autograd 详解 ===")
print()

# 1. 创建需要计算梯度的张量
x = torch.tensor([[1.0, 0.5]], requires_grad=True)
W1 = torch.tensor([[0.5, -0.3], [-0.2, 0.8]], requires_grad=True)
b1 = torch.tensor([0.1, -0.1], requires_grad=True)
W2 = torch.tensor([[0.4, 0.6]], requires_grad=True)
b2 = torch.tensor([0.2], requires_grad=True)

# 2. 前向传播（PyTorch自动记录计算图）
z1 = x @ W1.T + b1
a1 = torch.sigmoid(z1)
z2 = a1 @ W2.T + b2
y_true = torch.tensor([[0.8]])
loss = 0.5 * (z2 - y_true)**2

print(f"前向传播结果:")
print(f"  预测值 ŷ = {z2.item():.6f}")
print(f"  损失 L = {loss.item():.6f}")
print()

# 3. 反向传播
loss.backward()

print(f"反向传播后的梯度:")
print(f"  dL/dW1 = \n{W1.grad}")
print(f"  dL/db1 = {b1.grad}")
print(f"  dL/dW2 = {W2.grad}")
print(f"  dL/db2 = {b2.grad}")
print(f"  dL/dx = {x.grad}")
print()

# 4. 对比手动计算的梯度
print("对比手动计算的梯度（上一节）:")
print(f"  手动dW1 = \n{dW1}")
print(f"  PyTorch dW1 = \n{W1.grad.numpy()}")
print(f"  两者是否接近: {np.allclose(dW1, W1.grad.numpy(), atol=1e-4)}")

In [ ]:
# 可视化计算图
print("=== 计算图可视化 ===")
print()

# 创建一个简单的计算图并用ASCII展示
calc_graph = """
计算图结构（ASCII示意）：

    x ──────┐              
            ├─→ [@] ─→ [z1] ─→ [σ] ─→ [a1] ──┐
    W1 ────┘                                 │
    b1 ──────────────────────────────────────┘
                                              │
    W2 ────┐                                  │
           ├─→ [@] ─→ [z2] ─→ [-] ─→ [²] ─→ [L]
    b2 ────┘            ↑           
    y ──────────────────┘

其中：
  [@] = 矩阵乘法  
  [σ] = Sigmoid激活
  [-] = 减法
  [²] = 平方

反向传播方向（梯度从L往x传播）：
  dL/dL → dL/dz2 → dL/da1 → dL/dz1 → dL/dW1, dL/db1, dL/dx
                  → dL/dW2, dL/db2
"""
print(calc_graph)

## 四、梯度消失与梯度爆炸

### 梯度消失（Vanishing Gradient）

当网络很深时，梯度在反向传播过程中不断**乘以小于1的数**，导致：
- 浅层的梯度趋近于0
- 浅层的权重几乎不更新
- 网络"学不到"有用的浅层特征

**主要原因**：
1. Sigmoid/Tanh的导数最大值小于1
2. 权重初始化过小
3. 网络过深

### 梯度爆炸（Exploding Gradient）

相反地，梯度可能**乘以大于1的数**不断累积，导致：
- 梯度变得极大
- 参数更新幅度过大
- 训练不稳定

**主要原因**：
1. 权重初始化过大
2. RNN中长序列的连乘效应

In [ ]:
# 梯度消失演示
print("=== 梯度消失演示 ===")
print()

# 模拟10层网络的梯度传播
num_layers = 10
sigmoid_max_grad = 0.25  # sigmoid导数的最大值
weight_scale = 0.5       # 权重缩放

# 假设输出层梯度为1
grad_output = 1.0

print(f"假设输出层梯度 = {grad_output}")
print(f"每层梯度衰减因子 ≈ σ_max * W_scale = {sigmoid_max_grad} * {weight_scale} = {sigmoid_max_grad * weight_scale}")
print()
print(f"{'层数':<6} {'梯度值':>15} {'衰减倍数':>12}")
print("-" * 35)

gradients = [grad_output]
for layer in range(num_layers):
    grad_new = gradients[-1] * sigmoid_max_grad * weight_scale
    gradients.append(grad_new)

for i, g in enumerate(gradients):
    decay = g / gradients[0]
    print(f"L{num_layers-i:<4} {g:>15.2e} {decay:>12.2e}")

print()
print(f"经过{num_layers}层后，梯度衰减为原来的 {gradients[-1]/gradients[0]:.2e}")

# 可视化梯度衰减
fig, ax = plt.subplots(figsize=(10, 5))
layers = list(range(len(gradients)))
ax.bar(layers, gradients, color='steelblue', alpha=0.7)
ax.set_xlabel('Network Layer (from output to input)')
ax.set_ylabel('Gradient Magnitude')
ax.set_title(f'梯度消失演示 ({num_layers}层, σ+小权重)\n梯度从 {gradients[0]:.1f} 衰减到 {gradients[-1]:.2e}')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 梯度爆炸演示
print("=== 梯度爆炸演示 ===")
print()

num_layers = 10
weight_scale_large = 3.0   # 大权重初始化
grad_output = 0.01

print(f"假设输出层梯度 = {grad_output}")
print(f"每层梯度放大因子 ≈ W_scale = {weight_scale_large}")
print()
print(f"{'层数':<6} {'梯度值':>15} {'放大倍数':>12}")
print("-" * 35)

gradients_exp = [grad_output]
for layer in range(num_layers):
    grad_new = gradients_exp[-1] * weight_scale_large
    gradients_exp.append(grad_new)

for i, g in enumerate(gradients_exp):
    amplification = g / gradients_exp[0]
    print(f"L{num_layers-i:<4} {g:>15.2e} {amplification:>12.2e}")

print()
print(f"经过{num_layers}层后，梯度放大为原来的 {gradients_exp[-1]/gradients_exp[0]:.2e} 倍")

# 可视化梯度爆炸
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 线性坐标
layers = list(range(len(gradients_exp)))
axes[0].bar(layers, gradients_exp, color='red', alpha=0.7)
axes[0].set_xlabel('Network Layer (from output to input)')
axes[0].set_ylabel('Gradient Magnitude')
axes[0].set_title('Gradient Explosion (linear scale)')
axes[0].grid(True, alpha=0.3)

# 对数坐标
axes[1].bar(layers, gradients_exp, color='orange', alpha=0.7)
axes[1].set_xlabel('Network Layer (from output to input)')
axes[1].set_ylabel('梯度大小 (log)')
axes[1].set_title('Gradient Explosion (log scale)')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 五、解决梯度问题的方法

### 1. 使用ReLU激活函数

ReLU的导数：$f'(x) = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$

当 $x > 0$ 时导数为1，不会导致梯度衰减。

### 2. 合理的权重初始化

- **Xavier初始化**：适用于Sigmoid/Tanh
  $$W \sim N(0, \frac{1}{n_{in}})$$

- **He初始化**：适用于ReLU
  $$W \sim N(0, \frac{2}{n_{in}})$$

### 3. 梯度裁剪（Gradient Clipping）

当梯度范数超过阈值时，按比例缩小：

$$g_{\text{clipped}} = g \cdot \frac{\text{max_norm}}{\|g\|}$$

### 4. 残差连接（Residual Connection）

通过跳连让梯度可以直接传播：
$$y = F(x) + x$$

这是ResNet和Transformer的核心思想。

In [ ]:
# 梯度裁剪演示
print("=== 梯度裁剪演示 ===")
print()

# 创建一个有梯度爆炸倾向的模型
torch.manual_seed(42)

class ExplodingModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 大权重初始化
        self.fc = nn.Linear(10, 10)
        with torch.no_grad():
            self.fc.weight.data = torch.randn(10, 10) * 3.0  # 大权重
    
    def forward(self, x):
        return self.fc(x)

model = ExplodingModel()
criterion = nn.MSELoss()

# 创建一个大输入
x = torch.randn(1, 10) * 5
target = torch.randn(1, 10)

optimizer_no_clip = optim.SGD(model.parameters(), lr=0.01)
optimizer_clip = optim.SGD(model.parameters(), lr=0.01)

# 不用梯度裁剪
pred = model(x)
loss = criterion(pred, target)
optimizer_no_clip.zero_grad()
loss.backward()
grad_norm_no_clip = sum(p.grad.norm().item()**2 for p in model.parameters())**0.5
optimizer_no_clip.step()

# 用梯度裁剪
model2 = ExplodingModel()
pred2 = model2(x)
loss2 = criterion(pred2, target)
optimizer_clip.zero_grad()
loss2.backward()
torch.nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
grad_norm_clip = sum(p.grad.norm().item()**2 for p in model2.parameters())**0.5
optimizer_clip.step()

print(f"不用梯度裁剪:")
print(f"  梯度范数: {grad_norm_no_clip:.4f}")
print()
print(f"使用梯度裁剪 (max_norm=1.0):")
print(f"  裁剪后梯度范数: {grad_norm_clip:.4f}")
print()
print(f"梯度裁剪效果: {grad_norm_no_clip:.4f} → {grad_norm_clip:.4f} (减少了 {grad_norm_no_clip/grad_norm_clip:.1f} 倍)")

In [ ]:
# 不同初始化方法对比
print("\n=== 不同初始化方法对梯度流的影响 ===")
print()

def test_initialization(init_method, name):
    """测试不同初始化方法的梯度流"""
    torch.manual_seed(42)
    
    class TestNet(nn.Module):
        def __init__(self):
            super().__init__()
            layers = []
            sizes = [10, 32, 32, 32, 32, 32, 1]
            for i in range(len(sizes)-1):
                layer = nn.Linear(sizes[i], sizes[i+1])
                if init_method == 'xavier':
                    nn.init.xavier_normal_(layer.weight)
                elif init_method == 'he':
                    nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                elif init_method == 'large':
                    nn.init.normal_(layer.weight, std=3.0)
                elif init_method == 'small':
                    nn.init.normal_(layer.weight, std=0.01)
                layers.append(layer)
                if i < len(sizes)-2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.net(x)
    
    model = TestNet()
    x = torch.randn(1, 10)
    target = torch.randn(1, 1)
    
    output = model(x)
    loss = nn.MSELoss()(output, target)
    loss.backward()
    
    # 收集各层梯度
    layer_grads = []
    for name_p, param in model.named_parameters():
        if 'weight' in name_p and param.grad is not None:
            layer_grads.append(param.grad.norm().item())
    
    return layer_grads

# 测试不同初始化方法
methods = [
    ('xavier', 'Xavier初始化'),
    ('he', 'He初始化 (ReLU)'),
    ('small', '过小初始化 (std=0.01)'),
    ('large', '过大初始化 (std=3.0)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, (method, name) in enumerate(methods):
    grads = test_initialization(method, name)
    ax = axes[idx]
    ax.bar(range(len(grads)), grads, color='steelblue', alpha=0.7)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Gradient Norm')
    ax.set_title(f'{name}\n梯度范围: [{min(grads):.6f}, {max(grads):.6f}]')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 打印数值对比
print(f"{'初始化方法':<20} {'最小梯度':>12} {'最大梯度':>12} {'梯度比率':>12}")
print("-" * 60)
for method, name in methods:
    grads = test_initialization(method, name)
    ratio = max(grads) / (min(grads) + 1e-10)
    print(f"{name:<20} {min(grads):>12.6f} {max(grads):>12.6f} {ratio:>12.2f}")

## 六、本章总结

### 核心要点

1. **链式法则**是反向传播的数学基础
   - 从输出层往输入层，逐层应用链式法则
   - $\frac{\partial L}{\partial W^{[l]}} = \frac{\partial L}{\partial z^{[l]}} \cdot \frac{\partial z^{[l]}}{\partial W^{[l]}} = \delta^{[l]} \cdot (a^{[l-1]})^T$

2. **PyTorch Autograd**自动构建计算图并计算梯度
   - 前向传播记录操作
   - `.backward()`触发反向计算
   - 梯度存储在 `.grad` 中

3. **梯度消失**和**梯度爆炸**是深层网络的核心挑战
   - Sigmoid/Tanh导数小 → 梯度消失
   - 大权重初始化 → 梯度爆炸

4. **解决方案**：
   - ReLU激活（导数为1，不衰减）
   - He/Xavier初始化（保持梯度方差稳定）
   - 梯度裁剪（防止爆炸）
   - 残差连接（让梯度直接流动）

### 下一篇预告

下一篇将进入**RNN（循环神经网络）**的世界：
- RNN的基本结构与前向传播
- 通过时间反向传播（BPTT）
- RNN的梯度问题（为什么需要LSTM）